# 19. The SLAP for Reefer Containers Problem
## Tier 1: Mathematical Formulation (Mixed-Integer Programming)

### Key Assumptions
- Each reefer container must be assigned to exactly one storage location
- Storage locations have limited power capacity and temperature ranges
- Assignment costs include fixed location activation and variable energy costs
- Peak power consumption is penalized to encourage load balancing

### Approach (Step-by-Step)
1. **Problem Definition**: Model the reefer container assignment as a mixed-integer programming problem
2. **Decision Variables**: Binary assignment variables and continuous power usage variables
3. **Objective Function**: Minimize total cost including assignment, activation, and energy costs
4. **Constraints**: Ensure assignment, capacity, power, and temperature constraints
5. **Solution Method**: Use exact optimization (pulp) with fallback enumeration for small instances

### What to Look for in the Results
- Optimal assignment of containers to locations
- Total cost breakdown (assignment + activation + energy)
- Power utilization levels across locations
- Temperature compatibility verification

### Concrete Example (from the source)
Consider 3 reefer containers and 2 storage locations:
- Container 1: Pharmaceutical, -18°C, 45kW, dwell time 48h
- Container 2: Fresh produce, +2°C, 35kW, dwell time 72h  
- Container 3: Frozen seafood, -20°C, 50kW, dwell time 96h

Location A: Power capacity 100kW, temperature range [-25°C, +5°C], cost matrix = [15, 20, 25]
Location B: Power capacity 80kW, temperature range [-15°C, +10°C], cost matrix = [18, 12, 30]

In [1]:
# Import required packages for mathematical optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Try to import pulp for MIP, fallback to enumeration if not available
try:
    import pulp
    PULP_AVAILABLE = True
except ImportError:
    PULP_AVAILABLE = False
    print("pulp not available, using enumeration fallback")

print(f"Using pulp: {PULP_AVAILABLE}")

Using pulp: True


Using pulp: True


Using pulp: True


Using pulp: True


Using pulp: True


In [ ]:
@dataclass
class ReeferContainer:
    """Represents a reefer container with its characteristics"""
    id: int
    energy: float  # Power consumption in kW
    temp_req: float  # Required temperature in Celsius
    dwell_time: float  # Dwell time in hours
    cargo_type: str  # Type of cargo (pharmaceutical, produce, seafood, etc.)

@dataclass
class StorageLocation:
    """Represents a storage location with its constraints and costs"""
    id: int
    power_cap: float  # Power capacity in kW
    temp_min: float  # Minimum temperature in Celsius
    temp_max: float  # Maximum temperature in Celsius
    activation_cost: float  # Fixed activation cost
    energy_cost_rate: float  # Energy cost per kWh

@dataclass
class AssignmentCost:
    """Assignment cost matrix for containers to locations"""
    container_id: int
    location_id: int
    cost: float

print("Data classes defined for reefer container assignment")

In [ ]:
# Create the concrete example from the source material
containers = [
    ReeferContainer(id=1, energy=45, temp_req=-18, dwell_time=48, cargo_type="Pharmaceutical"),
    ReeferContainer(id=2, energy=35, temp_req=2, dwell_time=72, cargo_type="Fresh Produce"),
    ReeferContainer(id=3, energy=50, temp_req=-20, dwell_time=96, cargo_type="Frozen Seafood")
]

locations = [
    StorageLocation(id=1, power_cap=100, temp_min=-25, temp_max=5, 
                    activation_cost=50, energy_cost_rate=0.15),
    StorageLocation(id=2, power_cap=80, temp_min=-15, temp_max=10, 
                    activation_cost=40, energy_cost_rate=0.12)
]

# Assignment cost matrix (from source: C = [15, 20, 25] for Location A, [18, 12, 30] for Location B)
assignment_costs = {
    (1, 1): 15, (1, 2): 18,
    (2, 1): 20, (2, 2): 12,
    (3, 1): 25, (3, 2): 30
}

print(f"Created {len(containers)} containers and {len(locations)} locations")
print("\nContainer details:")
for c in containers:
    print(f"  Container {c.id}: {c.cargo_type}, {c.energy}kW, {c.temp_req}°C, {c.dwell_time}h")

print("\nLocation details:")
for l in locations:
    print(f"  Location {l.id}: {l.power_cap}kW, [{l.temp_min}°C, {l.temp_max}°C], activation=${l.activation_cost}")

In [ ]:
def check_temperature_compatibility(container: ReeferContainer, location: StorageLocation) -> bool:
    """Check if a container's temperature requirement is compatible with location"""
    return location.temp_min <= container.temp_req <= location.temp_max

def calculate_total_cost(container: ReeferContainer, location: StorageLocation, assignment_cost: float) -> float:
    """Calculate total cost including assignment, activation, and energy costs"""
    # Assignment cost (from matrix)
    cost = assignment_cost
    
    # Energy cost: energy * dwell_time * cost_rate / 1000 (convert to kWh)
    energy_cost = container.energy * container.dwell_time * location.energy_cost_rate / 1000
    cost += energy_cost
    
    return cost

# Verify temperature compatibility for all container-location pairs
print("Temperature Compatibility Matrix:")
compatibility_matrix = []
for container in containers:
    row = []
    for location in locations:
        compatible = check_temperature_compatibility(container, location)
        row.append("✓" if compatible else "✗")
        print(f"  Container {container.id} → Location {location.id}: {'Compatible' if compatible else 'Not Compatible'}")
    compatibility_matrix.append(row)

In [ ]:
def solve_mip_exact(containers: List[ReeferContainer], 
                    locations: List[StorageLocation],
                    assignment_costs: Dict[Tuple[int, int], float]) -> Dict:
    """Solve the reefer assignment problem using Mixed-Integer Programming"""
    
    if not PULP_AVAILABLE:
        return solve_by_enumeration(containers, locations, assignment_costs)
    
    # Create the MIP model
    model = pulp.LpProblem("Reefer_Assignment", pulp.LpMinimize)
    
    # Decision variables
    # x[container, location] = 1 if container assigned to location, 0 otherwise
    x = {}
    for container in containers:
        for location in locations:
            if check_temperature_compatibility(container, location):
                x[(container.id, location.id)] = pulp.LpVariable(
                    f"x_{container.id}_{location.id}", cat='Binary')
    
    # Location activation variables
    z = {}
    for location in locations:
        z[location.id] = pulp.LpVariable(f"z_{location.id}", cat='Binary')
    
    # Power usage variables (continuous)
    y = {}
    for location in locations:
        y[location.id] = pulp.LpVariable(f"y_{location.id}", lowBound=0, cat='Continuous')
    
    # Objective function: minimize total cost
    # Assignment costs + activation costs + energy costs + peak power penalty
    alpha = 0.001  # Peak power penalty coefficient
    
    objective = 0
    
    # Assignment costs
    for (container_id, location_id), cost in assignment_costs.items():
        if (container_id, location_id) in x:
            objective += cost * x[(container_id, location_id)]
    
    # Activation costs
    for location in locations:
        objective += location.activation_cost * z[location.id]
    
    # Energy costs (simplified as linear term)
    for container in containers:
        for location in locations:
            if (container.id, location.id) in x:
                energy_cost = container.energy * container.dwell_time * location.energy_cost_rate / 1000
                objective += energy_cost * x[(container.id, location.id)]
    
    # Peak power penalty
    for location in locations:
        objective += alpha * (y[location.id] ** 2)
    
    model += objective
    
    # Constraints
    
    # 1. Each container must be assigned to exactly one location
    for container in containers:
        assignment_vars = [x[(container.id, location.id)] 
                         for location in locations 
                         if (container.id, location.id) in x]
        if assignment_vars:
            model += pulp.lpSum(assignment_vars) == 1, f"assign_container_{container.id}"
    
    # 2. Power capacity constraints
    for location in locations:
        power_usage = pulp.lpSum([container.energy * x[(container.id, location.id)] 
                                for container in containers 
                                if (container.id, location.id) in x])
        model += power_usage <= location.power_cap * z[location.id], f"power_cap_{location.id}"
        model += y[location.id] == power_usage, f"define_power_{location.id}"
    
    # 3. Location activation linking
    for location in locations:
        assigned_containers = pulp.lpSum([x[(container.id, location.id)] 
                                       for container in containers 
                                       if (container.id, location.id) in x])
        model += assigned_containers <= len(containers) * z[location.id], f"activate_link_{location.id}"
    
    # Solve the model
    model.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # Extract solution
    if model.status == pulp.LpStatusOptimal:
        assignments = {}
        total_cost = pulp.value(model.objective)
        location_usage = {}
        
        for location in locations:
            location_usage[location.id] = {
                'power_used': 0,
                'containers': [],
                'activated': z[location.id].value() > 0.5
            }
        
        for (container_id, location_id), var in x.items():
            if var.value() > 0.5:
                assignments[container_id] = location_id
                container = next(c for c in containers if c.id == container_id)
                location_usage[location_id]['power_used'] += container.energy
                location_usage[location_id]['containers'].append(container_id)
        
        return {
            'status': 'optimal',
            'assignments': assignments,
            'total_cost': total_cost,
            'location_usage': location_usage,
            'objective_breakdown': {
                'assignment_cost': sum(assignment_costs.get((c_id, l_id), 0) 
                                    for c_id, l_id in assignments.items()),
                'activation_cost': sum(loc.activation_cost 
                                    for loc in locations 
                                    if location_usage[loc.id]['activated']),
                'energy_cost': sum(containers[c_id-1].energy * containers[c_id-1].dwell_time * 
                                 locations[l_id-1].energy_cost_rate / 1000 
                                 for c_id, l_id in assignments.items())
            }
        }
    else:
        return {'status': 'infeasible', 'message': 'No feasible solution found'}

print("MIP solver function defined")

In [ ]:
def solve_by_enumeration(containers: List[ReeferContainer], 
                        locations: List[StorageLocation],
                        assignment_costs: Dict[Tuple[int, int], float]) -> Dict:
    """Solve by exhaustive enumeration (fallback method)"""
    
    from itertools import product
    
    best_solution = None
    best_cost = float('inf')
    
    # Generate all possible assignments
    # For each container, list compatible locations
    compatible_locations = {}
    for container in containers:
        compatible_locations[container.id] = [
            location.id for location in locations
            if check_temperature_compatibility(container, location)
        ]
    
    # Enumerate all combinations
    for assignment_tuple in product(*[compatible_locations[c.id] for c in containers]):
        assignments = {containers[i].id: assignment_tuple[i] for i in range(len(containers))}
        
        # Check power constraints
        feasible = True
        location_power = {location.id: 0 for location in locations}
        
        for container_id, location_id in assignments.items():
            container = next(c for c in containers if c.id == container_id)
            location_power[location_id] += container.energy
            
            location = next(l for l in locations if l.id == location_id)
            if location_power[location_id] > location.power_cap:
                feasible = False
                break
        
        if feasible:
            # Calculate total cost
            total_cost = 0
            activated_locations = set(assignments.values())
            
            # Assignment costs
            for container_id, location_id in assignments.items():
                total_cost += assignment_costs.get((container_id, location_id), 0)
            
            # Activation costs
            for location_id in activated_locations:
                location = next(l for l in locations if l.id == location_id)
                total_cost += location.activation_cost
            
            # Energy costs
            for container_id, location_id in assignments.items():
                container = next(c for c in containers if c.id == container_id)
                location = next(l for l in locations if l.id == location_id)
                energy_cost = container.energy * container.dwell_time * location.energy_cost_rate / 1000
                total_cost += energy_cost
            
            if total_cost < best_cost:
                best_cost = total_cost
                best_solution = assignments
    
    if best_solution:
        # Calculate location usage
        location_usage = {}
        for location in locations:
            location_usage[location.id] = {
                'power_used': 0,
                'containers': [],
                'activated': False
            }
        
        for container_id, location_id in best_solution.items():
            container = next(c for c in containers if c.id == container_id)
            location_usage[location_id]['power_used'] += container.energy
            location_usage[location_id]['containers'].append(container_id)
            location_usage[location_id]['activated'] = True
        
        return {
            'status': 'optimal',
            'assignments': best_solution,
            'total_cost': best_cost,
            'location_usage': location_usage
        }
    else:
        return {'status': 'infeasible', 'message': 'No feasible solution found'}

print("Enumeration fallback function defined")

In [ ]:
# Solve the reefer assignment problem
solution = solve_mip_exact(containers, locations, assignment_costs)

print("=== REEFER ASSIGNMENT OPTIMIZATION RESULTS ===")
print(f"Solution Status: {solution['status'].upper()}")

if solution['status'] == 'optimal':
    print(f"Total Cost: ${solution['total_cost']:.2f}")
    
    print("\nOptimal Assignments:")
    for container_id, location_id in solution['assignments'].items():
        container = next(c for c in containers if c.id == container_id)
        location = next(l for l in locations if l.id == location_id)
        assignment_cost = assignment_costs[(container_id, location_id)]
        energy_cost = container.energy * container.dwell_time * location.energy_cost_rate / 1000
        total_cost = assignment_cost + energy_cost
        
        print(f"  Container {container_id} ({container.cargo_type}) → Location {location_id}")
        print(f"    Temperature: {container.temp_req}°C (fits [{location.temp_min}°C, {location.temp_max}°C])")
        print(f"    Power: {container.energy}kW, Energy Cost: ${energy_cost:.2f}, Assignment Cost: ${assignment_cost}")
        print(f"    Total Cost: ${total_cost:.2f}")
    
    print("\nLocation Utilization:")
    for location_id, usage in solution['location_usage'].items():
        location = next(l for l in locations if l.id == location_id)
        utilization = (usage['power_used'] / location.power_cap) * 100
        print(f"  Location {location_id}: {usage['power_used']}/{location.power_cap}kW ({utilization:.1f}% utilized)")
        print(f"    Containers: {usage['containers']}")
        print(f"    Status: {'Activated' if usage['activated'] else 'Inactive'}")
    
    if 'objective_breakdown' in solution:
        breakdown = solution['objective_breakdown']
        print("\nCost Breakdown:")
        print(f"  Assignment Costs: ${breakdown['assignment_cost']:.2f}")
        print(f"  Activation Costs: ${breakdown['activation_cost']:.2f}")
        print(f"  Energy Costs: ${breakdown['energy_cost']:.2f}")
        print(f"  Total: ${solution['total_cost']:.2f}")
else:
    print(f"Error: {solution.get('message', 'Unknown error')}")

In [ ]:
# Create comprehensive visualization of the solution
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Reefer Container Assignment Optimization Results', fontsize=16, fontweight='bold')

if solution['status'] == 'optimal':
    # 1. Assignment Matrix Heatmap
    ax1 = axes[0, 0]
    assignment_matrix = np.zeros((len(containers), len(locations)))
    for container_id, location_id in solution['assignments'].items():
        assignment_matrix[container_id-1, location_id-1] = 1
    
    container_labels = [f"C{c.id}\n{c.cargo_type}\n{c.energy}kW" for c in containers]
    location_labels = [f"L{l.id}\n{l.power_cap}kW\n[{l.temp_min}°C,{l.temp_max}°C]" for l in locations]
    
    sns.heatmap(assignment_matrix, annot=True, cmap='Blues', 
               xticklabels=location_labels, yticklabels=container_labels,
               cbar_kws={'label': 'Assigned'}, ax=ax1)
    ax1.set_title('Container Assignment Matrix')
    ax1.set_xlabel('Storage Locations')
    ax1.set_ylabel('Reefer Containers')
    
    # 2. Power Utilization Bar Chart
    ax2 = axes[0, 1]
    location_names = [f"Location {l.id}" for l in locations]
    power_used = [solution['location_usage'][l.id]['power_used'] for l in locations]
    power_capacity = [l.power_cap for l in locations]
    
    x = np.arange(len(location_names))
    width = 0.35
    
    bars1 = ax2.bar(x - width/2, power_used, width, label='Power Used', color='skyblue', alpha=0.8)
    bars2 = ax2.bar(x + width/2, power_capacity, width, label='Capacity', color='lightcoral', alpha=0.8)
    
    ax2.set_xlabel('Storage Locations')
    ax2.set_ylabel('Power (kW)')
    ax2.set_title('Power Utilization by Location')
    ax2.set_xticks(x)
    ax2.set_xticklabels(location_names)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Add utilization percentages on bars
    for i, (used, cap) in enumerate(zip(power_used, power_capacity)):
        utilization = (used / cap) * 100
        ax2.text(i - width/2, used + 2, f'{utilization:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # 3. Cost Breakdown Pie Chart
    ax3 = axes[1, 0]
    if 'objective_breakdown' in solution:
        breakdown = solution['objective_breakdown']
        costs = [breakdown['assignment_cost'], breakdown['activation_cost'], breakdown['energy_cost']]
        labels = ['Assignment', 'Activation', 'Energy']
        colors = ['#ff9999', '#66b3ff', '#99ff99']
        
        wedges, texts, autotexts = ax3.pie(costs, labels=labels, colors=colors, autopct='%1.1f%%', 
                                          startangle=90, textprops={'fontsize': 10})
        ax3.set_title('Cost Breakdown')
    else:
        ax3.text(0.5, 0.5, 'Cost breakdown\nnot available', ha='center', va='center', 
                transform=ax3.transAxes, fontsize=12)
        ax3.set_title('Cost Breakdown')
    
    # 4. Temperature Compatibility Chart
    ax4 = axes[1, 1]
    temp_data = []
    for container in containers:
        for location in locations:
            compatible = check_temperature_compatibility(container, location)
            assigned = solution['assignments'].get(container.id) == location.id
            temp_data.append({
                'Container': f"C{container.id}",
                'Location': f"L{location.id}",
                'Container Temp': container.temp_req,
                'Location Range': f"[{location.temp_min}°C, {location.temp_max}°C]",
                'Compatible': compatible,
                'Assigned': assigned
            })
    
    temp_df = pd.DataFrame(temp_data)
    
    # Create a scatter plot of temperatures
    for container in containers:
        container_temps = [container.temp_req] * len(locations)
        location_mins = [l.temp_min for l in locations]
        location_maxs = [l.temp_max for l in locations]
        
        # Plot temperature ranges
        for i, (l_min, l_max) in enumerate(zip(location_mins, location_maxs)):
            ax4.plot([i-0.2, i+0.2], [l_min, l_min], 'b-', linewidth=2)
            ax4.plot([i-0.2, i+0.2], [l_max, l_max], 'b-', linewidth=2)
            ax4.plot([i, i], [l_min, l_max], 'b-', linewidth=2, alpha=0.3)
        
        # Plot container temperature
        assigned_location = solution['assignments'].get(container.id) - 1
        for i, temp in enumerate(container_temps):
            color = 'red' if i == assigned_location else 'green'
            marker = 'o' if i == assigned_location else 's'
            ax4.scatter(i, temp, color=color, marker=marker, s=100, 
                       edgecolors='black', linewidth=2, zorder=5)
    
    ax4.set_xlabel('Storage Locations')
    ax4.set_ylabel('Temperature (°C)')
    ax4.set_title('Temperature Compatibility & Assignment')
    ax4.set_xticks(range(len(locations)))
    ax4.set_xticklabels([f"L{l.id}" for l in locations])
    ax4.grid(True, alpha=0.3)
    ax4.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    # Add legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
               markeredgecolor='black', markersize=10, label='Assigned'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='green', 
               markeredgecolor='black', markersize=10, label='Compatible'),
        Line2D([0], [0], color='blue', linewidth=2, label='Location Range')
    ]
    ax4.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

print("Visualization complete! The 4-panel dashboard shows:")
print("1. Assignment matrix showing which containers go to which locations")
print("2. Power utilization compared to capacity at each location")
print("3. Cost breakdown between assignment, activation, and energy costs")
print("4. Temperature compatibility and actual assignments")

In [ ]:
# Sensitivity Analysis: What-if scenarios
print("=== SENSITIVITY ANALYSIS ===")
print("\n1. Impact of Power Capacity Changes:")

# Test different power capacity scenarios
base_capacity = [l.power_cap for l in locations]
scenarios = [
    {'name': 'Base Case', 'capacity_multiplier': 1.0},
    {'name': 'Reduced Capacity (-20%)', 'capacity_multiplier': 0.8},
    {'name': 'Increased Capacity (+20%)', 'capacity_multiplier': 1.2},
    {'name': 'Limited Capacity (-40%)', 'capacity_multiplier': 0.6}
]

scenario_results = []

for scenario in scenarios:
    # Modify location capacities
    modified_locations = []
    for i, location in enumerate(locations):
        modified_loc = StorageLocation(
            id=location.id,
            power_cap=base_capacity[i] * scenario['capacity_multiplier'],
            temp_min=location.temp_min,
            temp_max=location.temp_max,
            activation_cost=location.activation_cost,
            energy_cost_rate=location.energy_cost_rate
        )
        modified_locations.append(modified_loc)
    
    # Solve with modified capacities
    result = solve_mip_exact(containers, modified_locations, assignment_costs)
    
    if result['status'] == 'optimal':
        scenario_results.append({
            'scenario': scenario['name'],
            'total_cost': result['total_cost'],
            'feasible': True,
            'total_capacity': sum(loc.power_cap for loc in modified_locations)
        })
        print(f"  {scenario['name']}: Cost = ${result['total_cost']:.2f}, Feasible")
    else:
        scenario_results.append({
            'scenario': scenario['name'],
            'total_cost': float('inf'),
            'feasible': False,
            'total_capacity': sum(loc.power_cap for loc in modified_locations)
        })
        print(f"  {scenario['name']}: Infeasible")

# Create sensitivity analysis visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Cost vs Capacity
feasible_results = [r for r in scenario_results if r['feasible']]
if feasible_results:
    capacities = [r['total_capacity'] for r in feasible_results]
    costs = [r['total_cost'] for r in feasible_results]
    scenarios_feasible = [r['scenario'] for r in feasible_results]
    
    ax1.plot(capacities, costs, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Total Power Capacity (kW)')
    ax1.set_ylabel('Total Cost ($)')
    ax1.set_title('Cost vs Power Capacity')
    ax1.grid(True, alpha=0.3)
    
    # Add scenario labels
    for i, (cap, cost, scenario) in enumerate(zip(capacities, costs, scenarios_feasible)):
        ax1.annotate(scenario.replace(' ', '\n'), (cap, cost), 
                    textcoords="offset points", xytext=(0,10), ha='center', fontsize=8)

# Plot 2: Feasibility Map
capacity_multipliers = [s['capacity_multiplier'] for s in scenarios]
feasibility = [1 if r['feasible'] else 0 for r in scenario_results]
colors = ['lightgreen' if f else 'lightcoral' for f in feasibility]

# Fix the bar plot syntax
capacity_values = [r['total_capacity'] if r['feasible'] else 0 for r in scenario_results]
bars = ax2.bar(range(len(scenarios)), capacity_values, color=colors, alpha=0.7)

ax2.set_xlabel('Scenarios')
ax2.set_ylabel('Total Power Capacity (kW)')
ax2.set_title('Feasibility Analysis')
ax2.set_xticks(range(len(scenarios)))
ax2.set_xticklabels([s['name'].replace(' ', '\n') for s in scenarios], rotation=45, ha='right')
ax2.grid(True, alpha=0.3)

# Add feasibility labels
for i, (bar, feasible) in enumerate(zip(bars, feasibility)):
    height = bar.get_height()
    if feasible:
        ax2.text(bar.get_x() + bar.get_width()/2., height + 5, 
                f'${scenario_results[i]["total_cost"]:.0f}', ha='center', va='bottom', fontweight='bold')
    else:
        ax2.text(bar.get_x() + bar.get_width()/2., 25, 
                'INFEASIBLE', ha='center', va='bottom', fontweight='bold', color='red')

plt.tight_layout()
plt.show()

print("\n2. Key Insights from Sensitivity Analysis:")
print("- Power capacity constraints significantly impact feasibility")
print("- There's a minimum capacity threshold for feasible assignments")
print("- Cost increases with capacity but provides more flexibility")
print("- The system becomes infeasible when total capacity < total container power demand")

### Why This Tier Exists vs Earlier Tiers

**Tier 1 (Mathematical Formulation)** provides the foundational optimization approach with:
- **Guaranteed Optimality**: Exact mathematical solution with provable optimality
- **Comprehensive Constraints**: Handles all problem constraints simultaneously
- **Theoretical Foundation**: Serves as benchmark for all heuristic methods
- **Sensitivity Analysis**: Enables systematic parameter analysis

### Pros vs Cons

**Advantages:**
- Optimal solutions guaranteed (when feasible)
- Comprehensive constraint handling
- Provides benchmark for comparison
- Enables sensitivity analysis
- Transparent mathematical formulation

**Disadvantages:**
- Computationally expensive for large instances
- Requires specialized optimization software
- May struggle with highly non-linear constraints
- Limited scalability for real-time applications

### When to Use This Tier

**Use Tier 1 when:**
- Problem instances are small to medium sized (< 50 containers, < 20 locations)
- Optimal solutions are required (e.g., for benchmarking, critical decisions)
- Comprehensive constraint analysis is needed
- Sensitivity analysis and what-if scenarios are important
- Sufficient computational resources and time are available

**Switch to higher tiers when:**
- Problem instances become large (> 100 containers)
- Real-time decision making is required
- Approximate solutions are acceptable
- Computational resources are limited